# mpH2MM / smFRET Analysis — PTU files (Alexa555 / Alexa647)

**Donor:** Alexa 555 (excitation ~520 nm, emission ~570 nm)  
**Acceptor:** Alexa 647 (excitation ~640 nm, emission ~690 nm)  

Pipeline:
1. Convert `.ptu` → Photon-HDF5 (`.h5`) via `phconvert`
2. Load and burst-search with `FRETbursts`
3. First-round H²MM filtering (remove donor-only / acceptor-only bursts)
4. Second-round H²MM fitting on FRET-active bursts
5. Dwell-time / ES-plot visualisation

## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import fretbursts as frb
import phconvert as phc
import bursth2mm as bhm   # mpH2MM high-level interface

print('NumPy      :', np.__version__)
print('FRETbursts :', frb.__version__)
print('phconvert  :', phc.__version__)
print('bursth2mm  :', bhm.__version__)

## 2 · Helper functions

In [ ]:
# ---------------------------------------------------------------------------
# PTU → Photon-HDF5 conversion
# ---------------------------------------------------------------------------
# PTU files produced by PicoQuant software (SymPhoTime, etc.) are
# self-contained: all timing metadata lives inside the file header.
# phconvert.pqreader reads them directly — no separate .set file needed.
#
# Python 3.12 note: np.bool / np.int / np.float / np.complex were removed;
# use the plain built-ins (bool, int, float, complex) or np.bool_ etc.
# ---------------------------------------------------------------------------

def ptu_to_hdf5(
    ptu_file: str,
    output_path: str,
    description: str = '',
    author: str = '',
    sample_name: str = '',
    buffer_name: str = '',
    dye_names: str = 'Alexa555, Alexa647',
):
    """
    Convert a PicoQuant .ptu file to Photon-HDF5 format.

    The function reads the PTU with phconvert.pqreader, sets up the
    correct metadata for a 2-colour ALEX experiment using
    Alexa555 (donor, ch=1) / Alexa647 (acceptor, ch=0), and writes
    a Photon-HDF5 file ready for FRETbursts.

    Parameters
    ----------
    ptu_file    : path to the input .ptu file
    output_path : path for the output .h5 file
    description : free-text experiment description
    author      : experimenter name
    sample_name : sample identifier
    buffer_name : buffer / solution identifier
    dye_names   : comma-separated dye names (default 'Alexa555, Alexa647')
    """

    # --- Read PTU --------------------------------------------------------
    # pqreader.load_ptfile returns
    #   (timestamps, detectors, nanotimes, metadata_dict)
    print(f'Reading {ptu_file} ...')
    timestamps, detectors, nanotimes, meta = phc.pqreader.load_ptfile(ptu_file)

    # Timestamps unit is stored in the PTU header (seconds / tick)
    timestamps_unit = meta['tags']['MeasDesc_GlobalResolution']['value']

    # TCSPC parameters
    tcspc_unit    = meta['tags']['MeasDesc_Resolution']['value']          # s / bin
    tcspc_num_bins = int(meta['tags']['TTResultFormat_BitsPerRecord']['value'])  # usually 4096 / 1024
    # Fallback: derive num_bins from nanotimes range
    if nanotimes is not None:
        tcspc_num_bins = int(nanotimes.max()) + 1
    tcspc_range = tcspc_unit * tcspc_num_bins

    # Laser repetition rate — stored in header as Hz
    # Some PTU headers use 'TTResult_SyncRate'; others use
    # 'MeasDesc_GlobalResolution' already in seconds.
    # We compute the period (seconds) from GlobalResolution.
    laser_rep_rate = 1.0 / timestamps_unit   # Hz  (ticks-per-second = 1 / unit)
    # Safer: look for explicit sync-rate tag
    if 'TTResult_SyncRate' in meta.get('tags', {}):
        laser_rep_rate = float(meta['tags']['TTResult_SyncRate']['value'])

    # --- Diagnostic: show detector histogram and TCSPC decays -----------
    unique_det, counts = np.unique(detectors, return_counts=True)
    print('Detectors found:', dict(zip(unique_det.tolist(), counts.tolist())))
    print(f'timestamps_unit = {timestamps_unit:.3e} s')
    print(f'tcspc_unit      = {tcspc_unit:.3e} s   ({tcspc_num_bins} bins)')

    if nanotimes is not None:
        fig, ax = plt.subplots(figsize=(7, 3))
        for d in unique_det:
            mask = detectors == d
            ax.hist(nanotimes[mask], bins=256, histtype='step', label=f'det {d}')
        ax.set_yscale('log')
        ax.set_xlabel('TCSPC bin')
        ax.set_ylabel('counts')
        ax.legend()
        ax.set_title('TCSPC decays by detector')
        plt.tight_layout()
        plt.show()

    # --- Build Photon-HDF5 metadata dicts --------------------------------
    #
    # ALEX period convention (in TCSPC-bins or timestamp ticks — must match
    # what FRETbursts expects for alex_period_donor / alex_period_acceptor).
    #
    # Your setup:
    #   donor excitation window  : 0 – 1000  ticks
    #   acceptor excitation window: 1000 – 2000 ticks
    #   donor    detector channel: 1  (Alexa555 emission)
    #   acceptor detector channel: 0  (Alexa647 emission)
    #
    # Adjust alex_excitation_period1/2 if your PTU uses a different
    # period range (check the TCSPC-decay histogram above).

    measurement_specs = dict(
        measurement_type='smFRET-usALEX',
        alex_excitation_period1=(0, 1000),        # donor   excitation window
        alex_excitation_period2=(1000, 2000),      # acceptor excitation window
        detectors_specs=dict(
            spectral_ch1=[1],   # donor   emission  → Alexa555  (det 1)
            spectral_ch2=[0],   # acceptor emission → Alexa647  (det 0)
        ),
        laser_repetition_rate=laser_rep_rate,
    )

    setup = dict(
        num_pixels=2,
        num_spots=1,
        num_spectral_ch=2,
        num_polarization_ch=1,
        num_split_ch=1,
        modulated_excitation=True,
        excitation_alternated=[True, True],
        excitation_cw=[False, False],
        lifetime=(nanotimes is not None),
        laser_repetition_rates=[laser_rep_rate],
        # Alexa555 ex 520 nm / em 570 nm ; Alexa647 ex 640 nm / em 690 nm
        excitation_wavelengths=[520e-9, 640e-9],
        detection_wavelengths=[570e-9, 690e-9],
    )

    identity = dict(
        author=author,
        author_affiliation='',
    )

    sample = dict(
        sample_name=sample_name,
        buffer_name=buffer_name,
        dye_names=dye_names,
    )

    provenance = dict(
        filename=ptu_file,
        creation_time=str(meta.get('tags', {}).get('File_CreatingTime', {}).get('value', '')),
    )

    # --- Assemble photon_data dict ----------------------------------------
    photon_data = dict(
        timestamps=timestamps,
        timestamps_specs={'timestamps_unit': timestamps_unit},
        detectors=detectors,
        measurement_specs=measurement_specs,
    )

    if nanotimes is not None:
        photon_data['nanotimes'] = nanotimes
        photon_data['nanotimes_specs'] = dict(
            tcspc_unit=tcspc_unit,
            tcspc_range=tcspc_range,
            tcspc_num_bins=tcspc_num_bins,
        )

    h5_data = dict(
        identity=identity,
        sample=sample,
        provenance=provenance,
        description=description,
        photon_data=photon_data,
        setup=setup,
    )

    # --- Write HDF5 -------------------------------------------------------
    phc.hdf5.save_photon_hdf5(
        h5_data,
        h5_fname=output_path,
        overwrite=True,
        close=True,
    )
    print(f'Saved Photon-HDF5 → {output_path}')

In [ ]:
# ---------------------------------------------------------------------------
# FRETbursts burst search + selection
# ---------------------------------------------------------------------------
# Adjust the burst-search and selection thresholds as needed.
# Key parameters:
#   D_ON  : timestamp range (ticks) for donor-excitation period
#   A_ON  : timestamp range (ticks) for acceptor-excitation period
#           These must match the alex_excitation_period values above.
#   m     : number of consecutive photons used to compute local rate
#   F     : burst-search threshold (multiples of background rate)
#   th1   : minimum burst size (Dex photons) for first selection cut
#   th1   : minimum naa (Aex-Aem photons) for second selection cut
# ---------------------------------------------------------------------------

def load_and_burst_search(
    h5_file: str,
    D_ON: tuple = (0, 1000),
    A_ON: tuple = (1000, 2000),
    m: int = 20,
    F: float = 7.0,
    min_burst_size: int = 60,
    min_naa: int = 40,
):
    """
    Load a Photon-HDF5 file, apply ALEX periods, run burst search
    and two-stage burst selection.

    Returns
    -------
    data     : full FRETbursts Data object (all bursts)
    data_sel : burst-selected Data object
    """
    data = frb.loader.photon_hdf5(h5_file)

    # Apply ALEX donor / acceptor excitation windows
    # donor=1 → Alexa555 emission is on detector 1
    # acceptor=0 → Alexa647 emission is on detector 0
    data.add(
        donor=1,
        acceptor=0,
        alex_period_donor=D_ON,
        alex_period_acceptor=A_ON,
    )
    frb.loader.alex_apply_period(data)

    # Background estimation
    data.calc_bg(fun=frb.bg.exp_fit, time_s=30, F_bg=1.7)

    # All-photon burst search
    data.burst_search(m=m, F=F)
    data.fuse_bursts(ms=0)

    # Selection 1: minimum total Dex photons
    data_sel = data.select_bursts(
        frb.select_bursts.size,
        add_naa=False,
        th1=min_burst_size,
    )

    # Selection 2: minimum Aex-Aem photons (stoichiometry filter)
    data_sel = data_sel.select_bursts(
        frb.select_bursts.naa,
        th1=min_naa,
    )

    return data, data_sel

In [ ]:
# ---------------------------------------------------------------------------
# BVA (Burst Variance Analysis)
# ---------------------------------------------------------------------------

def BVA(data, chunk_size: int = 5):
    """
    Burst Variance Analysis: compute sub-burst FRET std-dev.

    Parameters
    ----------
    data       : FRETbursts Data object with burst selection applied
    chunk_size : photons per sub-burst window

    Returns
    -------
    E_eff : list of arrays  — mean FRET per burst
    std_E : list of arrays  — std-dev of FRET per burst
    """
    E_eff, std_E = [], []
    for ich, mburst in enumerate(data.mburst):
        stdE, E = [], []
        Aem = data.get_ph_mask(ich=ich, ph_sel=frb.Ph_sel(Dex='Aem'))
        Dex = data.get_ph_mask(ich=ich, ph_sel=frb.Ph_sel(Dex='DAem'))
        for istart, istop in zip(mburst.istart, mburst.istop):
            phots = Aem[istart:istop + 1][Dex[istart:istop + 1]]
            Esub = [
                phots[nb:ne].sum()
                for nb, ne in zip(
                    range(0, phots.size, chunk_size),
                    range(chunk_size, phots.size + 1, chunk_size),
                )
            ]
            stdE.append(np.std(Esub) / chunk_size)
            E.append(sum(Esub) / (len(Esub) * chunk_size) if Esub else np.nan)
        E_eff.append(np.array(E))
        std_E.append(np.array(stdE))
    return E_eff, std_E


def bin_bva(E, std, R: int = 10, B_thr: int = 50):
    """Bin BVA results for a smoother plot."""
    E_cat   = np.concatenate(E)
    std_cat = np.concatenate(std)
    bn = np.linspace(0, 1, R + 1)
    std_avg, E_avg = np.empty(R), np.empty(R)
    for i, (bb, be) in enumerate(zip(bn[:-1], bn[1:])):
        mask = (bb <= E_cat) & (E_cat < be)
        if mask.sum() > B_thr:
            std_avg[i] = np.mean(std_cat[mask])
            E_avg[i]   = np.mean(E_cat[mask])
        else:
            std_avg[i] = E_avg[i] = -1
    return E_avg, std_avg


# BVA theoretical shot-noise curve settings
n_bva      = 5          # sub-burst chunk size
x_T        = np.arange(0, 1.01, 0.01)
y_T        = np.sqrt((x_T * (1 - x_T)) / n_bva)

In [ ]:
# ---------------------------------------------------------------------------
# ALEX-2CDE filter
# ---------------------------------------------------------------------------

def apply_alex2cde(
    frb_data,
    tau_s: float = 150e-6,
    contrast: float = 150,
    alex_2cde_cutoff: float = 95,
):
    """
    Apply the ALEX-2CDE filter to remove donor-only / acceptor-only artefacts.

    Returns a new FRETbursts Data object with the filter applied.
    """
    tau = int(tau_s / frb_data.clk_p)   # convert to raw timestamp units

    ph      = frb_data.ph_times_m[0]
    bursts  = frb_data.mburst[0]

    ph_dex   = frb_data.get_ph_times(ph_sel=frb.Ph_sel(Dex='DAem'))
    ph_aex   = frb_data.get_ph_times(ph_sel=frb.Ph_sel(Aex='Aem'))
    mask_dex = frb_data.get_ph_mask(ph_sel=frb.Ph_sel(Dex='DAem'))
    mask_aex = frb_data.get_ph_mask(ph_sel=frb.Ph_sel(Aex='Aem'))

    KDE_DexTi = frb.phtools.phrates.kde_laplace(ph_dex, tau, time_axis=ph)
    KDE_AexTi = frb.phtools.phrates.kde_laplace(ph_aex, tau, time_axis=ph)

    ALEX_2CDE, BRDex, BRAex = [], [], []
    for ib, burst in enumerate(bursts):
        sl = slice(int(burst.istart), int(burst.istop) + 1)
        if not mask_dex[sl].any() or not mask_aex[sl].any():
            ALEX_2CDE.append(np.nan)
            continue

        kde_dexdex = KDE_DexTi[sl][mask_dex[sl]]
        kde_aexdex = KDE_AexTi[sl][mask_dex[sl]]
        N_chaex    = mask_aex[sl].sum()
        BRDex.append(np.sum(kde_aexdex / kde_dexdex) / N_chaex)

        kde_aexaex = KDE_AexTi[sl][mask_aex[sl]]
        kde_dexaex = KDE_DexTi[sl][mask_aex[sl]]
        N_chdex    = mask_dex[sl].sum()
        BRAex.append(np.sum(kde_dexaex / kde_aexaex) / N_chdex)

        ALEX_2CDE.append(100 - contrast * (BRDex[-1] - BRAex[-1]))

    ALEX_2CDE = np.array(ALEX_2CDE)

    plt.figure(figsize=(5, 4))
    plt.hist2d(frb_data.E[0], ALEX_2CDE, bins=50, range=((-0.2, 1.2), (0, 150)))
    plt.xlabel('E')
    plt.ylabel('ALEX-2CDE')
    plt.title(f'ALEX-2CDE  τ = {tau_s * 1e6:.0f} µs')
    plt.tight_layout()
    plt.show()

    valid = np.isfinite(ALEX_2CDE)
    masks = [valid & (ALEX_2CDE > alex_2cde_cutoff) & (frb_data.S[0] < 0.5)]
    return frb_data.select_bursts_mask_apply(masks)

## 3 · Convert PTU → Photon-HDF5

Set `ptu_file` to your `.ptu` file path.  
The converted `.h5` file will be written to `output_h5`.

> **Multiple files?** Convert each PTU separately, then use
> `frb.loader.photon_hdf5` with a list of paths, or concatenate
> timestamps manually before saving (see the comment block below).

In [ ]:
# --- USER INPUT -----------------------------------------------------------
ptu_file  = r'C:/path/to/your_measurement.ptu'   # ← change this
output_h5 = 'my_smfret_measurement.h5'            # ← output HDF5 filename
# --------------------------------------------------------------------------

ptu_to_hdf5(
    ptu_file=ptu_file,
    output_path=output_h5,
    description='smFRET measurement — Alexa555 / Alexa647',
    author='',                    # your name
    sample_name='',               # e.g. 'DNA ruler 10 bp'
    buffer_name='',               # e.g. 'PBS pH 7.4'
    dye_names='Alexa555, Alexa647',
)

## 4 · Load HDF5, burst search, and selection

If you already have a Photon-HDF5 file (e.g. exported from SymPhoTime
or previously converted), set `output_h5` above and run from here.

In [ ]:
smfret_data, smfret_data_sel = load_and_burst_search(
    h5_file=output_h5,
    D_ON=(0, 1000),      # donor excitation ALEX window   (ticks)
    A_ON=(1000, 2000),   # acceptor excitation ALEX window (ticks)
    m=20,                # min consecutive photons for burst start
    F=7.0,               # burst threshold (× background rate)
    min_burst_size=60,   # min Dex photons per burst
    min_naa=40,          # min Aex-Aem photons per burst
)

In [ ]:
print(f'Acquisition duration   : {smfret_data.acquisition_duration:.1f} s')
print(f'Total bursts           : {smfret_data.mburst[0].num_bursts}')
print(f'Bursts after selection : {smfret_data_sel.mburst[0].num_bursts}')
print(f'Burst rate             : {smfret_data.mburst[0].num_bursts / smfret_data.acquisition_duration:.2f} /s')

In [ ]:
# ALEX joint plot (E vs S)
frb.alex_jointplot(smfret_data_sel)
plt.show()

## 5 · First-round H²MM fit

Fit models with up to `max_state` states; use ICL / BICp to pick
the optimal number of states, then filter out donor-only and
acceptor-only bursts.

In [ ]:
smfret_data_mp = bhm.BurstData(smfret_data_sel)

In [ ]:
smfret_data_mp.models.calc_models(
    max_state=6,
    max_iter=3600,
)

In [ ]:
# ICL criterion — sets the ideal model automatically
fig, ax = plt.subplots(figsize=(4, 3))
smfret_data_mp.models.find_ideal('ICL', auto_set=True)
bhm.ICL_plot(smfret_data_mp.models, highlight_ideal=True)
plt.tight_layout()
plt.show()

# BICp criterion — for comparison
fig, ax = plt.subplots(figsize=(4, 3))
smfret_data_mp.models.find_ideal('BICp', auto_set=False)
bhm.BICp_plot(smfret_data_mp.models, highlight_ideal=True)
plt.axhline(0.005, color='red', linestyle='--', label='threshold 0.005')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Inspect the automatically selected model
model_id_round1 = 3   # ← adjust if the criterion suggests a different number

plt.figure(figsize=(5, 4))
bhm.dwell_ES_scatter(smfret_data_mp.models[model_id_round1], alpha=0.2, s=2)
bhm.trans_arrow_ES(smfret_data_mp.models[model_id_round1])
plt.tight_layout()
plt.show()

print(smfret_data_mp.models[model_id_round1])

## 6 · Select FRET-active dwells

Burst-type bitmask:
- `0b0010` (2)  → low-FRET dwell only
- `0b1000` (8)  → high-FRET dwell only
- `0b1010` (10) → both low- and high-FRET dwells in same burst

In [ ]:
fret_active_mask = (
    (smfret_data_mp.models[model_id_round1].burst_type == int(0b0010)) |
    (smfret_data_mp.models[model_id_round1].burst_type == int(0b1000)) |
    (smfret_data_mp.models[model_id_round1].burst_type == int(0b1010))
)
lf_hf_idxs = np.where(fret_active_mask)

n_before = len(smfret_data_sel.E[0])
n_after  = len(smfret_data_sel.E[0][lf_hf_idxs[0]])
print(f'{n_after} of {n_before} bursts remaining after H²MM filtering.')

fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(smfret_data_sel.E[0],             bins=50, density=True,
        histtype='step', label='All bursts')
ax.hist(smfret_data_sel.E[0][lf_hf_idxs[0]], bins=50, density=True,
        histtype='step', label='H²MM filtered')
ax.set_xlim(0, 1)
ax.set_xlabel(r'$E$')
ax.set_ylabel('Norm. density')
ax.legend()
plt.tight_layout()
plt.show()

## 7 · Second-round H²MM fit (FRET-active bursts only)

In [ ]:
smfret_data_hmm = smfret_data_sel.select_bursts_mask_apply(lf_hf_idxs)

frb.alex_jointplot(smfret_data_hmm)
plt.show()

In [ ]:
smfret_data_hmm_mp = bhm.BurstData(smfret_data_hmm)

smfret_data_hmm_mp.models.calc_models(
    max_state=6,
    max_iter=3600,
)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
smfret_data_hmm_mp.models.find_ideal('ICL', auto_set=True)
bhm.ICL_plot(smfret_data_hmm_mp.models, highlight_ideal=True)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(4, 3))
smfret_data_hmm_mp.models.find_ideal('BICp', auto_set=False)
bhm.BICp_plot(smfret_data_hmm_mp.models, highlight_ideal=True)
plt.axhline(0.005, color='red', linestyle='--')
plt.tight_layout()
plt.show()

## 8 · Plot H²MM results

Set `model_id_final` to the number of states suggested by ICL / BICp above.

In [ ]:
model_id_final = 2   # ← adjust to ICL/BICp suggestion

fig, ax = plt.subplots(figsize=(5, 4))
bhm.dwell_ES_scatter(smfret_data_hmm_mp.models[model_id_final], alpha=0.6, s=1)
bhm.trans_arrow_ES(smfret_data_hmm_mp.models[model_id_final], min_rate=0)
bhm.scatter_ES(smfret_data_hmm_mp.models[model_id_final], s=100, c='r')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel(r'$E$')
ax.set_ylabel(r'$S$')
plt.tight_layout()
plt.show()

print(smfret_data_hmm_mp.models[model_id_final])

## 9 · Dwell-time extraction

`bursth2mm` stores dwell times and state assignments in the model object.
The cells below extract dwell times per state and plot cumulative
survival functions (useful for fitting rate constants).

In [ ]:
model = smfret_data_hmm_mp.models[model_id_final]

# dwell_pos  : array of (burst_index, start_index, stop_index, state_id)
# dwell_dur  : dwell duration in number of photons
# Access via model.dwell_pos and model.dwell_dur (attribute names may
# differ slightly across bursth2mm versions — check model.__dict__ if needed)
try:
    dwell_pos = model.dwell_pos
    print('dwell_pos shape:', dwell_pos.shape)
    print('Columns: burst_idx, ph_start, ph_stop, state')
    print(dwell_pos[:5])
except AttributeError:
    print('dwell_pos not found — check model attributes:', [k for k in model.__dict__ if 'dwell' in k.lower()])

In [ ]:
# Convert dwell durations (photons) to seconds using mean burst photon rate
# A simple estimate: mean_ph_rate = total_photons / acquisition_duration
#
# For a more accurate per-burst rate, divide by the burst duration in seconds.
# bursth2mm >= 0.3 exposes model.dwell_times_s directly — use that if available.

num_states = model_id_final

fig, axes = plt.subplots(1, num_states, figsize=(4 * num_states, 4), sharey=True)
if num_states == 1:
    axes = [axes]

for state_id, ax in enumerate(axes):
    try:
        # bursth2mm >= 0.3: dwell_times returns list of arrays per state
        dwell_t = model.dwell_times[state_id]     # seconds
        xlabel  = 'Dwell time (s)'
    except (AttributeError, IndexError):
        # Fallback: use photon counts as proxy for dwell time
        dwell_t = dwell_pos[dwell_pos[:, 3] == state_id, 2] \
                - dwell_pos[dwell_pos[:, 3] == state_id, 1]
        xlabel  = 'Dwell duration (photons)'

    dwell_t = dwell_t[dwell_t > 0]

    # Survival function (1 - CDF)
    t_sorted  = np.sort(dwell_t)
    survival  = 1 - np.arange(1, len(t_sorted) + 1) / len(t_sorted)

    ax.semilogy(t_sorted, survival, '.', markersize=3)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Survival probability')
    ax.set_title(f'State {state_id}  (n={len(dwell_t)})')

plt.suptitle('Dwell-time survival functions per H²MM state', y=1.02)
plt.tight_layout()
plt.show()

## 10 · BVA (optional)

Burst Variance Analysis complements H²MM by confirming that FRET
heterogeneity exceeds shot noise.

In [ ]:
E_bva, std_bva = BVA(smfret_data_hmm, chunk_size=n_bva)
E_avg, std_avg = bin_bva(E_bva, std_bva, R=10, B_thr=50)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(np.concatenate(E_bva), np.concatenate(std_bva),
           s=2, alpha=0.3, color='grey', label='per burst')
valid = E_avg > 0
ax.scatter(E_avg[valid], std_avg[valid], s=40, color='blue',
           zorder=5, label='binned average')
ax.plot(x_T, y_T, 'k--', label='shot noise')
ax.set_xlabel(r'$\langle E \rangle$')
ax.set_ylabel(r'$\sigma_E$')
ax.legend()
plt.tight_layout()
plt.show()